<a href="https://colab.research.google.com/github/JoaoChoma/taess_2026/blob/main/estudos_experimentais/helio_dacio/dataset/jpetclinic/i_toMicroservices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json

with open("P1-Reflective-Architect_jpetstore.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Keys in meta:", list(meta.keys()))
print("Keys in execucoes[0]:", list(meta["execucoes"][0].keys()))
print("Any other files?", [f for f in os.listdir('.') if f.endswith('.json') or f.endswith('.csv') or f.endswith('.zip')])

In [ ]:
import numpy as np
import pandas as pd
import json

with open("P1-Reflective-Architect_jpetstore.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

zstar = np.array(meta["fronteira_referencia"])
z_ed = np.linalg.norm(zstar, axis=1)
df = pd.DataFrame(meta["execucoes"])

# Check which runs have their ed_min in z_ed!
matches = []
for idx, row in df.iterrows():
    diffs = np.abs(z_ed - row["ed_min"])
    min_d = diffs.min()
    best_z_idx = diffs.argmin()
    if min_d < 1e-9:
        matches.append((row["config"], row["repetition"], row["ed_min"], best_z_idx, zstar[best_z_idx]))
print(f"Exact ed_min matches in Z*: {len(matches)} / 40 runs")
for m in matches:
    print(f"  {m[0]:28s} rep {m[1]}: ed_min={m[2]:.6f} -> Z*[{m[3]}] = {np.round(m[4], 4)}")

In [ ]:
print("Top 5 smallest ED in Z*:")
idx_sorted = np.argsort(z_ed)
for rank, i in enumerate(idx_sorted[:10], 1):
    print(f"  Rank {rank:2d}: Z*[{i:3d}] ED={z_ed[i]:.6f} -> {np.round(zstar[i], 4)}")

In [ ]:
print("Top 5 smallest ED in Z*:")
idx_sorted = np.argsort(z_ed)
for rank, i in enumerate(idx_sorted[:10], 1):
    print(f"  Rank {rank:2d}: Z*[{i:3d}] ED={z_ed[i]:.6f} -> {np.round(zstar[i], 4)}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Set publication style (IEEE/ACM Transactions)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.9

with open("P1-Reflective-Architect_jpetstore.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

df = pd.DataFrame(meta["execucoes"])

config_order = [
    'nsgaiii__elit_on__penalty',
    'nsgaiii__elit_off__penalty',
    'rnsgaiii__elit_on__penalty',
    'rnsgaiii__elit_off__penalty',
    'nsgaiii__elit_on__kill',
    'nsgaiii__elit_off__kill',
    'rnsgaiii__elit_on__kill',
    'rnsgaiii__elit_off__kill',
]
short_labels = [
    'C1\nNSGA\nON|PEN',
    'C2\nNSGA\nOFF|PEN',
    'C3\nR-NSGA\nON|PEN',
    'C4*\nR-NSGA\nOFF|PEN',
    'C5\nNSGA\nON|KILL',
    'C6\nNSGA\nOFF|KILL',
    'C7\nR-NSGA\nON|KILL',
    'C8\nR-NSGA\nOFF|KILL',
]
cid_map = {k: f"C{i+1}" for i, k in enumerate(config_order)}
df["cid"] = pd.Categorical(df["config"].map(cid_map), categories=[f"C{i}" for i in range(1, 9)], ordered=True)
df = df.sort_values(["cid", "repetition"]).reset_index(drop=True)

# Okabe-Ito colorblind-safe palette:
# PENALTY: #0072B2 (Blue), C4 highlight: #009E73 (Bluish Green), KILL: #D55E00 (Vermilion)
box_colors = ['#56B4E9', '#0072B2', '#56B4E9', '#009E73', '#E69F00', '#D55E00', '#E69F00', '#D55E00']

# ==============================================================================
# FIGURA 1: Boxplots + Stripplots (n=5) + Painel Resumo Fatorial (2x3)
# ==============================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10.5), dpi=300)
fig.suptitle(
    "Figura 1 — Distribuição dos Indicadores de Qualidade por Configuração (P1-Reflective-Architect × JPetStore, N = 40, n = 5)",
    fontsize=13.5, fontweight='bold', y=0.98
)

panels = [
    (axes[0, 0], "hv", "Hypervolume (HV) [↑ Maior é Melhor]", "Kruskal-Wallis: H(7) = 17.8712, p = 0.0126* | ANOVA: F = 3.7612, p = 0.0044**"),
    (axes[0, 1], "igd_plus", "IGD+ contra Z* [↓ Menor é Melhor]", "Kruskal-Wallis: H(7) = 11.2068, p = 0.1298 (NS) | F3 SS% = 15.46% (p = 0.0124*)"),
    (axes[0, 2], "ed_media", "Distância Euclidiana Média ao Ideal [↓ Menor é Melhor]", "Kruskal-Wallis: H(7) = 10.4283, p = 0.1656 (NS) | F1 SS% = 13.72% (p = 0.0219*)"),
    (axes[1, 0], "ed_min", "Distância Euclidiana Mínima ao Ideal [↓ Menor é Melhor]", "ANOVA One-Way: F(7,32) = 2.1082, p = 0.0713 (NS) | F3 SS% = 16.96% (p = 0.0082**)"),
    (axes[1, 1], "spacing", "Spacing (Uniformidade / Compacidade) [↓ Menor]", "ANOVA One-Way: F(7,32) = 6.3276, p = 0.000105*** | Kruskal: H = 29.5054***"),
]

np.random.seed(42)
for ax, col, title, subtitle in panels:
    data_groups = [df.loc[df["config"] == k, col].to_numpy() for k in config_order]
    bp = ax.boxplot(
        data_groups, positions=np.arange(8), widths=0.52, patch_artist=True,
        showmeans=True, meanprops=dict(marker='D', markerfacecolor='white', markeredgecolor='black', markersize=5.5),
        medianprops=dict(color='#111111', linewidth=2.0),
        flierprops=dict(marker='o', markersize=4, alpha=0.5),
        boxprops=dict(linewidth=1.1, edgecolor='#222222'),
        whiskerprops=dict(linewidth=1.1, color='#222222'),
        capprops=dict(linewidth=1.1, color='#222222')
    )
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(box_colors[i])
        patch.set_alpha(0.88 if i == 3 else 0.68)
        if i == 3:
            patch.set_edgecolor('#004D38')
            patch.set_linewidth(2.0)

    # Jittered stripplot (n=5 points)
    for i, vals in enumerate(data_groups):
        jitter = np.random.uniform(-0.14, 0.14, size=len(vals))
        ax.scatter(
            np.full(len(vals), i) + jitter, vals,
            color=box_colors[i], edgecolor='black', linewidth=0.9, s=42, zorder=4, alpha=0.95
        )

    # Vertical divider between PENALTY (C1..C4) and KILL (C5..C8)
    ax.axvline(3.5, color='#555555', linestyle='--', linewidth=1.1, alpha=0.7)
    ax.set_xticks(np.arange(8))
    ax.set_xticklabels(short_labels, fontsize=8.2, fontweight='bold')
    ax.set_title(f"{title}\n{subtitle}", fontsize=9.5, fontweight='bold', pad=8)
    ax.grid(axis='y', linestyle=':', alpha=0.35, color='#666666')
    sns.despine(ax=ax)

    # Highlight C4 background band
    ax.axvspan(2.62, 3.38, color='#009E73', alpha=0.08, zorder=0)

    # Add FDR-BH significance brackets on HV
    if col == "hv":
        y_max = df["hv"].max()
        ax.set_ylim(0.58, y_max + 0.16)
        # Bracket C4 (idx 3) vs C7 (idx 6)
        y1 = y_max + 0.035
        ax.plot([3, 3, 6, 6], [y1 - 0.015, y1, y1, y1 - 0.015], lw=1.3, c='#004D38')
        ax.text(4.5, y1 + 0.005, "Dunn FDR-BH p = 0.0488* (Â12 = 1.00)", ha='center', va='bottom', fontsize=8, fontweight='bold', color='#004D38')
        # Bracket C4 (idx 3) vs C8 (idx 7)
        y2 = y_max + 0.095
        ax.plot([3, 3, 7, 7], [y2 - 0.015, y2, y2, y2 - 0.015], lw=1.3, c='#004D38')
        ax.text(5.0, y2 + 0.005, "Dunn FDR-BH p = 0.0434* (Â12 = 1.00)", ha='center', va='bottom', fontsize=8, fontweight='bold', color='#004D38')
    elif col == "igd_plus":
        ax.annotate("Líder Global: C4\n(Med=0.0880, Rbar=6.40)", xy=(3, 0.034), xytext=(1.2, 0.025),
                    arrowprops=dict(arrowstyle="->", color='#004D38', lw=1.2),
                    fontsize=8, fontweight='bold', color='#004D38')
    elif col == "ed_media":
        ax.annotate("Mínima Dispersão: C4\n(Med=0.5551, Rbar=7.60)", xy=(3, 0.511), xytext=(0.4, 0.49),
                    arrowprops=dict(arrowstyle="->", color='#004D38', lw=1.2),
                    fontsize=8, fontweight='bold', color='#004D38')
    elif col == "spacing":
        ax.text(1.5, 0.142, "Bloco PENALTY\n(Fronteira Expandida)", ha='center', va='top', fontsize=8.2, color='#0072B2', fontweight='bold')
        ax.text(5.5, 0.142, "Bloco KILL (13 pares p_BH ≤ 0.05)\n(Fronteira Colapsada/Estreita)", ha='center', va='top', fontsize=8.2, color='#D55E00', fontweight='bold')

# Panel [1, 2] (bottom-right): Legend & Summary Rank Table
ax_leg = axes[1, 2]
ax_leg.axis('off')

legend_patches = [
    mpatches.Patch(facecolor='#009E73', edgecolor='#004D38', linewidth=1.8, label='C4 ( Campeã Global ): R-NSGA-III | ELIT_OFF | PENALTY'),
    mpatches.Patch(facecolor='#56B4E9', edgecolor='#222222', label='PENALTY + ELIT_ON (C1: NSGA-III, C3: R-NSGA-III)'),
    mpatches.Patch(facecolor='#0072B2', edgecolor='#222222', label='PENALTY + ELIT_OFF (C2: NSGA-III)'),
    mpatches.Patch(facecolor='#E69F00', edgecolor='#222222', label='KILL + ELIT_ON (C5: NSGA-III, C7: R-NSGA-III)'),
    mpatches.Patch(facecolor='#D55E00', edgecolor='#222222', label='KILL + ELIT_OFF (C6: NSGA-III, C8: R-NSGA-III)'),
]
leg = ax_leg.legend(handles=legend_patches, loc='upper center', frameon=True, facecolor='#F8F9FA', edgecolor='#CCCCCC', fontsize=8.8, title="Codificação Fatorial 2×2×2 (◇ = Média | Traço = Mediana)", title_fontsize=9.2)

# Build summary table of Medians and Rbar
table_data = [
    ["C1 (NSGA|ON|PEN)",  "0.9947 (29.2)", "0.1274 (18.0)", "0.6428 (25.8)", "0.4559 (18.8)", "0.0934 (33.8)"],
    ["C2 (NSGA|OFF|PEN)", "0.9250 (24.0)", "0.1486 (20.6)", "0.6767 (24.8)", "0.4762 (22.4)", "0.0627 (27.8)"],
    ["C3 (RNSG|ON|PEN)",  "0.9448 (23.2)", "0.1532 (19.4)", "0.6342 (22.6)", "0.4339 (17.6)", "0.0784 (31.2)"],
    ["C4* (RNSG|OFF|PEN)","1.0551 (33.4)", "0.0880 ( 6.4)", "0.5551 ( 7.6)", "0.3339 ( 7.0)", "0.0561 (28.4)"],
    ["C5 (NSGA|ON|KILL)", "0.8304 (14.8)", "0.1668 (23.6)", "0.6555 (25.6)", "0.4703 (22.8)", "0.0349 (13.2)"],
    ["C6 (NSGA|OFF|KILL)","0.8669 (17.6)", "0.1489 (21.8)", "0.6446 (22.6)", "0.4738 (24.0)", "0.0327 (10.4)"],
    ["C7 (RNSG|ON|KILL)", "0.8054 (11.8)", "0.1852 (28.0)", "0.6474 (20.6)", "0.4988 (26.8)", "0.0349 (11.4)"],
    ["C8 (RNSG|OFF|KILL)","0.7680 (10.0)", "0.1874 (26.2)", "0.5617 (14.4)", "0.4952 (24.6)", "0.0309 ( 7.8)"],
]
col_headers = ["Configuração", "HV Med (R̄)", "IGD+ Med (R̄)", "ED_m Med (R̄)", "ED_min (R̄)", "Spac Med (R̄)"]

tbl = ax_leg.table(
    cellText=table_data, colLabels=col_headers,
    loc='bottom', bbox=[0.0, 0.0, 1.0, 0.58], cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7.7)
for (row_idx, col_idx), cell in tbl.get_celld().items():
    cell.set_edgecolor('#CCCCCC')
    if row_idx == 0:
        cell.set_facecolor('#2B3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row_idx == 4:  # C4 row
        cell.set_facecolor('#D1F2EB')
        cell.set_text_props(fontweight='bold', color='#004D38')
    elif row_idx in [1, 2, 3]:
        cell.set_facecolor('#EBF5FB')
    else:
        cell.set_facecolor('#FDEDEC')

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("fig1_distribuicao_indicadores.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig1_distribuicao_indicadores.png:", os.path.getsize("fig1_distribuicao_indicadores.png"), "bytes")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 11.5), dpi=300)
fig.suptitle(
    "Figura 1 — Distribuição dos Indicadores de Qualidade por Configuração (P1-Reflective-Architect × JPetStore, N = 40, n = 5)",
    fontsize=13.5, fontweight='bold', y=0.98
)

short_labels_clean = [
    'C1 (PEN)\nNSGA|ON',
    'C2 (PEN)\nNSGA|OFF',
    'C3 (PEN)\nR-NSGA|ON',
    'C4* (PEN)\nR-NSGA|OFF',
    'C5 (KILL)\nNSGA|ON',
    'C6 (KILL)\nNSGA|OFF',
    'C7 (KILL)\nR-NSGA|ON',
    'C8 (KILL)\nR-NSGA|OFF',
]

panels = [
    (axes[0, 0], "hv", "Hypervolume (HV) [↑ Maior é Melhor]", "Kruskal H(7) = 17.8712, p = 0.0126* | ANOVA F = 3.7612, p = 0.0044**"),
    (axes[0, 1], "igd_plus", "IGD+ contra Z* [↓ Menor é Melhor]", "Kruskal H(7) = 11.2068, p = 0.1298 (NS) | F3 SS% = 15.46% (p = 0.0124*)"),
    (axes[0, 2], "ed_media", "Distância Euclidiana Média ao Ideal [↓ Menor]", "Kruskal H(7) = 10.4283, p = 0.1656 (NS) | F1 SS% = 13.72% (p = 0.0219*)"),
    (axes[1, 0], "ed_min", "Distância Euclidiana Mínima ao Ideal [↓ Menor]", "ANOVA F(7,32) = 2.1082, p = 0.0713 (NS) | F3 SS% = 16.96% (p = 0.0082**)"),
    (axes[1, 1], "spacing", "Spacing (Uniformidade / Compacidade) [↓ Menor]", "ANOVA F(7,32) = 6.3276, p = 0.000105*** | Kruskal H = 29.5054***"),
]

np.random.seed(42)
for ax, col, title, subtitle in panels:
    data_groups = [df.loc[df["config"] == k, col].to_numpy() for k in config_order]
    bp = ax.boxplot(
        data_groups, positions=np.arange(8), widths=0.50, patch_artist=True,
        showmeans=True, meanprops=dict(marker='D', markerfacecolor='white', markeredgecolor='black', markersize=5.5),
        medianprops=dict(color='#111111', linewidth=2.0),
        flierprops=dict(marker='o', markersize=3.5, alpha=0.4),
        boxprops=dict(linewidth=1.1, edgecolor='#222222'),
        whiskerprops=dict(linewidth=1.1, color='#222222'),
        capprops=dict(linewidth=1.1, color='#222222')
    )
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(box_colors[i])
        patch.set_alpha(0.88 if i == 3 else 0.68)
        if i == 3:
            patch.set_edgecolor('#004D38')
            patch.set_linewidth(2.0)

    for i, vals in enumerate(data_groups):
        jitter = np.random.uniform(-0.14, 0.14, size=len(vals))
        ax.scatter(
            np.full(len(vals), i) + jitter, vals,
            color=box_colors[i], edgecolor='black', linewidth=0.9, s=42, zorder=4, alpha=0.95
        )

    ax.axvline(3.5, color='#555555', linestyle='--', linewidth=1.1, alpha=0.7)
    ax.set_xticks(np.arange(8))
    ax.set_xticklabels(short_labels_clean, fontsize=7.6, fontweight='bold')
    ax.set_title(f"{title}\n{subtitle}", fontsize=9.0, fontweight='bold', pad=8)
    ax.grid(axis='y', linestyle=':', alpha=0.35, color='#666666')
    sns.despine(ax=ax)
    ax.axvspan(2.62, 3.38, color='#009E73', alpha=0.08, zorder=0)

    if col == "hv":
        y_max = df["hv"].max()
        ax.set_ylim(0.58, y_max + 0.17)
        y1 = y_max + 0.035
        ax.plot([3, 3, 6, 6], [y1 - 0.015, y1, y1, y1 - 0.015], lw=1.3, c='#004D38')
        ax.text(4.5, y1 + 0.005, "Dunn FDR-BH p = 0.0488* (Â12 = 1.00)", ha='center', va='bottom', fontsize=7.8, fontweight='bold', color='#004D38')
        y2 = y_max + 0.098
        ax.plot([3, 3, 7, 7], [y2 - 0.015, y2, y2, y2 - 0.015], lw=1.3, c='#004D38')
        ax.text(5.0, y2 + 0.005, "Dunn FDR-BH p = 0.0434* (Â12 = 1.00)", ha='center', va='bottom', fontsize=7.8, fontweight='bold', color='#004D38')
    elif col == "igd_plus":
        ax.set_ylim(0.005, 0.285)
        ax.annotate("Líder Global: C4 (Med=0.0880, R̄=6.40)", xy=(3, 0.034), xytext=(3.6, 0.015),
                    arrowprops=dict(arrowstyle="->", color='#004D38', lw=1.2),
                    fontsize=7.8, fontweight='bold', color='#004D38')
    elif col == "ed_media":
        ax.set_ylim(0.46, 0.88)
        ax.annotate("Mínima Dispersão: C4 (Med=0.5551, R̄=7.60)", xy=(3, 0.511), xytext=(0.1, 0.472),
                    arrowprops=dict(arrowstyle="->", color='#004D38', lw=1.2),
                    fontsize=7.8, fontweight='bold', color='#004D38')
    elif col == "ed_min":
        ax.set_ylim(0.27, 0.63)
        ax.annotate("Ótimo Global Z*: C4 Rep 3 (0.2998)", xy=(3, 0.300), xytext=(3.6, 0.282),
                    arrowprops=dict(arrowstyle="->", color='#004D38', lw=1.2),
                    fontsize=7.8, fontweight='bold', color='#004D38')
    elif col == "spacing":
        ax.set_ylim(0.015, 0.162)
        ax.text(1.5, 0.155, "Bloco PENALTY\n(Fronteira Expandida)", ha='center', va='top', fontsize=8.0, color='#0072B2', fontweight='bold')
        ax.text(5.5, 0.155, "Bloco KILL (13 pares p_BH ≤ 0.05)\n(Fronteira Colapsada/Estreita)", ha='center', va='top', fontsize=8.0, color='#D55E00', fontweight='bold')

ax_leg = axes[1, 2]
ax_leg.axis('off')
leg = ax_leg.legend(handles=legend_patches, loc='upper center', frameon=True, facecolor='#F8F9FA', edgecolor='#CCCCCC', fontsize=8.5, title="Codificação Fatorial 2×2×2 (◇ = Média | Traço = Mediana)", title_fontsize=9.0)

tbl = ax_leg.table(
    cellText=table_data, colLabels=col_headers,
    loc='bottom', bbox=[0.0, 0.0, 1.0, 0.58], cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7.6)
for (row_idx, col_idx), cell in tbl.get_celld().items():
    cell.set_edgecolor('#CCCCCC')
    if row_idx == 0:
        cell.set_facecolor('#2B3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row_idx == 4:
        cell.set_facecolor('#D1F2EB')
        cell.set_text_props(fontweight='bold', color='#004D38')
    elif row_idx in [1, 2, 3]:
        cell.set_facecolor('#EBF5FB')
    else:
        cell.set_facecolor('#FDEDEC')

plt.subplots_adjust(left=0.05, right=0.98, top=0.90, bottom=0.06, hspace=0.36, wspace=0.20)
fig.savefig("fig1_distribuicao_indicadores.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Regenerated fig1_distribuicao_indicadores.png:", os.path.getsize("fig1_distribuicao_indicadores.png"), "bytes")

In [ ]:
from scipy import stats

def orientar(x, ind):
    x = np.asarray(x, dtype=float)
    return x if ind == "hv" else -x

def a12_val(a, b):
    a = np.asarray(a, dtype=float)[:, None]
    b = np.asarray(b, dtype=float)[None, :]
    cmp = a - b
    return float((np.sum(cmp > 0) + 0.5 * np.sum(cmp == 0)) / cmp.size)

def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    ordem = np.argsort(p)
    ranq = p[ordem] * n / (np.arange(n) + 1)
    ranq = np.minimum.accumulate(ranq[::-1])[::-1]
    out = np.empty(n, dtype=float)
    out[ordem] = np.clip(ranq, 0.0, 1.0)
    return out

def compute_matrices(df, ind, config_order):
    # 8x8 A12, p_dunn_raw, p_dunn_bh
    mat_a12 = np.full((8, 8), 0.50)
    mat_praw = np.ones((8, 8))
    mat_pbh = np.ones((8, 8))

    valores = df[ind].to_numpy(dtype=float)
    rotulos = df["config"].to_numpy()
    N = len(valores)
    ranks = stats.rankdata(valores)
    _, counts = np.unique(valores, return_counts=True)
    T = np.sum(counts**3 - counts)
    sigma2 = N * (N + 1) / 12.0 - T / (12.0 * (N - 1))

    pairs = []
    p_raw_list = []
    for i in range(8):
        for j in range(i + 1, 8):
            cA, cB = config_order[i], config_order[j]
            ra = ranks[rotulos == cA]
            rb = ranks[rotulos == cB]
            denom = np.sqrt(sigma2 * (1.0 / len(ra) + 1.0 / len(rb)))
            z = (ra.mean() - rb.mean()) / denom
            p = float(2.0 * stats.norm.sf(abs(z)))
            pairs.append((i, j))
            p_raw_list.append(p)

    p_bh_list = bh_fdr(p_raw_list)
    for (i, j), pr, pb in zip(pairs, p_raw_list, p_bh_list):
        mat_praw[i, j] = mat_praw[j, i] = pr
        mat_pbh[i, j] = mat_pbh[j, i] = pb

    for i in range(8):
        for j in range(8):
            if i != j:
                xa = orientar(df.loc[df["config"] == config_order[i], ind], ind)
                xb = orientar(df.loc[df["config"] == config_order[j], ind], ind)
                mat_a12[i, j] = a12_val(xa, xb)

    return mat_a12, mat_praw, mat_pbh

fig, axes = plt.subplots(2, 2, figsize=(16, 13.5), dpi=300)
fig.suptitle(
    "Figura 2 — Matrizes 8×8 de Tamanho de Efeito Vargha-Delaney (Â12) e Significância Pós-Hoc de Dunn\n"
    "Célula (Linha i, Coluna j) = P(Config i supera Config j) | ** p_FDR-BH ≤ 0.05 | * p_bruto ≤ 0.05",
    fontsize=13.0, fontweight='bold', y=0.98
)

axis_labels = [
    "C1 (NSGA|ON|PEN)",
    "C2 (NSGA|OFF|PEN)",
    "C3 (R-NSGA|ON|PEN)",
    "C4* (R-NSGA|OFF|PEN)",
    "C5 (NSGA|ON|KILL)",
    "C6 (NSGA|OFF|KILL)",
    "C7 (R-NSGA|ON|KILL)",
    "C8 (R-NSGA|OFF|KILL)",
]

hm_specs = [
    (axes[0, 0], "hv", "Hypervolume (HV — ↑ Maior é Melhor)\n14 Pares Efeito Grande | 2 Pares p_FDR-BH ≤ 0.05 (C4 vs C8, C4 vs C7)"),
    (axes[0, 1], "igd_plus", "IGD+ contra Z* (↓ Menor é Melhor — Orientado)\n10 Pares Efeito Grande | C4 supera todas as 7 configs (Â12 ∈ [0.76, 1.00])"),
    (axes[1, 0], "ed_media", "Distância Euclidiana Média ao Ideal (↓ Menor é Melhor — Orientado)\n10 Pares Efeito Grande | C4 (PEN) e C8 (KILL) lideram seus blocos"),
    (axes[1, 1], "spacing", "Spacing / Uniformidade (↓ Menor — Orientado)\n19 Pares Efeito Grande | 13 Pares p_FDR-BH ≤ 0.05 (Colapso Espacial do Bloco KILL)"),
]

for ax, ind, title in hm_specs:
    mat_a12, mat_praw, mat_pbh = compute_matrices(df, ind, config_order)
    annot_mat = np.empty((8, 8), dtype=object)
    for i in range(8):
        for j in range(8):
            if i == j:
                annot_mat[i, j] = "0.50"
            else:
                stars = "**" if mat_pbh[i, j] <= 0.05 else ("*" if mat_praw[i, j] <= 0.05 else "")
                annot_mat[i, j] = f"{mat_a12[i, j]:.2f}{stars}"

    sns.heatmap(
        mat_a12, ax=ax, cmap="RdBu_r", vmin=0.0, vmax=1.0, center=0.50,
        annot=annot_mat, fmt="", cbar=True,
        cbar_kws={"shrink": 0.82, "label": "Vargha-Delaney Â12 (Linha supera Coluna)"},
        xticklabels=axis_labels, yticklabels=axis_labels,
        linewidths=0.8, linecolor='white',
        annot_kws={"fontsize": 8.8, "fontweight": "bold"}
    )
    # Block dividers (PENALTY vs KILL)
    ax.axhline(4.0, color='#111111', linewidth=2.2, linestyle='--')
    ax.axvline(4.0, color='#111111', linewidth=2.2, linestyle='--')
    # Highlight C4 row (row index 3)
    ax.add_patch(plt.Rectangle((0, 3), 8, 1, fill=False, edgecolor='#00FFAA', lw=2.5, clip_on=False))

    ax.set_title(title, fontsize=10.2, fontweight='bold', pad=10)
    ax.set_xticklabels(axis_labels, rotation=32, ha='right', fontsize=8.3, fontweight='bold')
    ax.set_yticklabels(axis_labels, rotation=0, fontsize=8.3, fontweight='bold')
    ax.set_xlabel("Configuração B (Coluna)", fontsize=9.0, fontweight='bold')
    ax.set_ylabel("Configuração A (Linha)", fontsize=9.0, fontweight='bold')

plt.subplots_adjust(left=0.12, right=0.96, top=0.90, bottom=0.09, hspace=0.36, wspace=0.28)
fig.savefig("fig2_heatmaps_efeito_A12_fdr.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig2_heatmaps_efeito_A12_fdr.png:", os.path.getsize("fig2_heatmaps_efeito_A12_fdr.png"), "bytes")

In [ ]:
from scipy import stats

def orientar(x, ind):
    x = np.asarray(x, dtype=float)
    return x if ind == "hv" else -x

def a12_val(a, b):
    a = np.asarray(a, dtype=float)[:, None]
    b = np.asarray(b, dtype=float)[None, :]
    cmp = a - b
    return float((np.sum(cmp > 0) + 0.5 * np.sum(cmp == 0)) / cmp.size)

def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    ordem = np.argsort(p)
    ranq = p[ordem] * n / (np.arange(n) + 1)
    ranq = np.minimum.accumulate(ranq[::-1])[::-1]
    out = np.empty(n, dtype=float)
    out[ordem] = np.clip(ranq, 0.0, 1.0)
    return out

def compute_matrices(df, ind, config_order):
    # 8x8 A12, p_dunn_raw, p_dunn_bh
    mat_a12 = np.full((8, 8), 0.50)
    mat_praw = np.ones((8, 8))
    mat_pbh = np.ones((8, 8))

    valores = df[ind].to_numpy(dtype=float)
    rotulos = df["config"].to_numpy()
    N = len(valores)
    ranks = stats.rankdata(valores)
    _, counts = np.unique(valores, return_counts=True)
    T = np.sum(counts**3 - counts)
    sigma2 = N * (N + 1) / 12.0 - T / (12.0 * (N - 1))

    pairs = []
    p_raw_list = []
    for i in range(8):
        for j in range(i + 1, 8):
            cA, cB = config_order[i], config_order[j]
            ra = ranks[rotulos == cA]
            rb = ranks[rotulos == cB]
            denom = np.sqrt(sigma2 * (1.0 / len(ra) + 1.0 / len(rb)))
            z = (ra.mean() - rb.mean()) / denom
            p = float(2.0 * stats.norm.sf(abs(z)))
            pairs.append((i, j))
            p_raw_list.append(p)

    p_bh_list = bh_fdr(p_raw_list)
    for (i, j), pr, pb in zip(pairs, p_raw_list, p_bh_list):
        mat_praw[i, j] = mat_praw[j, i] = pr
        mat_pbh[i, j] = mat_pbh[j, i] = pb

    for i in range(8):
        for j in range(8):
            if i != j:
                xa = orientar(df.loc[df["config"] == config_order[i], ind], ind)
                xb = orientar(df.loc[df["config"] == config_order[j], ind], ind)
                mat_a12[i, j] = a12_val(xa, xb)

    return mat_a12, mat_praw, mat_pbh

fig, axes = plt.subplots(2, 2, figsize=(16, 13.5), dpi=300)
fig.suptitle(
    "Figura 2 — Matrizes 8×8 de Tamanho de Efeito Vargha-Delaney (Â12) e Significância Pós-Hoc de Dunn\n"
    "Célula (Linha i, Coluna j) = P(Config i supera Config j) | ** p_FDR-BH ≤ 0.05 | * p_bruto ≤ 0.05",
    fontsize=13.0, fontweight='bold', y=0.98
)

axis_labels = [
    "C1 (NSGA|ON|PEN)",
    "C2 (NSGA|OFF|PEN)",
    "C3 (R-NSGA|ON|PEN)",
    "C4* (R-NSGA|OFF|PEN)",
    "C5 (NSGA|ON|KILL)",
    "C6 (NSGA|OFF|KILL)",
    "C7 (R-NSGA|ON|KILL)",
    "C8 (R-NSGA|OFF|KILL)",
]

hm_specs = [
    (axes[0, 0], "hv", "Hypervolume (HV — ↑ Maior é Melhor)\n14 Pares Efeito Grande | 2 Pares p_FDR-BH ≤ 0.05 (C4 vs C8, C4 vs C7)"),
    (axes[0, 1], "igd_plus", "IGD+ contra Z* (↓ Menor é Melhor — Orientado)\n10 Pares Efeito Grande | C4 supera todas as 7 configs (Â12 ∈ [0.76, 1.00])"),
    (axes[1, 0], "ed_media", "Distância Euclidiana Média ao Ideal (↓ Menor é Melhor — Orientado)\n10 Pares Efeito Grande | C4 (PEN) e C8 (KILL) lideram seus blocos"),
    (axes[1, 1], "spacing", "Spacing / Uniformidade (↓ Menor — Orientado)\n19 Pares Efeito Grande | 13 Pares p_FDR-BH ≤ 0.05 (Colapso Espacial do Bloco KILL)"),
]

for ax, ind, title in hm_specs:
    mat_a12, mat_praw, mat_pbh = compute_matrices(df, ind, config_order)
    annot_mat = np.empty((8, 8), dtype=object)
    for i in range(8):
        for j in range(8):
            if i == j:
                annot_mat[i, j] = "0.50"
            else:
                stars = "**" if mat_pbh[i, j] <= 0.05 else ("*" if mat_praw[i, j] <= 0.05 else "")
                annot_mat[i, j] = f"{mat_a12[i, j]:.2f}{stars}"

    sns.heatmap(
        mat_a12, ax=ax, cmap="RdBu_r", vmin=0.0, vmax=1.0, center=0.50,
        annot=annot_mat, fmt="", cbar=True,
        cbar_kws={"shrink": 0.82, "label": "Vargha-Delaney Â12 (Linha supera Coluna)"},
        xticklabels=axis_labels, yticklabels=axis_labels,
        linewidths=0.8, linecolor='white',
        annot_kws={"fontsize": 8.8, "fontweight": "bold"}
    )
    # Block dividers (PENALTY vs KILL)
    ax.axhline(4.0, color='#111111', linewidth=2.2, linestyle='--')
    ax.axvline(4.0, color='#111111', linewidth=2.2, linestyle='--')
    # Highlight C4 row (row index 3)
    ax.add_patch(plt.Rectangle((0, 3), 8, 1, fill=False, edgecolor='#00FFAA', lw=2.5, clip_on=False))

    ax.set_title(title, fontsize=10.2, fontweight='bold', pad=10)
    ax.set_xticklabels(axis_labels, rotation=32, ha='right', fontsize=8.3, fontweight='bold')
    ax.set_yticklabels(axis_labels, rotation=0, fontsize=8.3, fontweight='bold')
    ax.set_xlabel("Configuração B (Coluna)", fontsize=9.0, fontweight='bold')
    ax.set_ylabel("Configuração A (Linha)", fontsize=9.0, fontweight='bold')

plt.subplots_adjust(left=0.12, right=0.96, top=0.90, bottom=0.09, hspace=0.36, wspace=0.28)
fig.savefig("fig2_heatmaps_efeito_A12_fdr.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig2_heatmaps_efeito_A12_fdr.png:", os.path.getsize("fig2_heatmaps_efeito_A12_fdr.png"), "bytes")

In [ ]:
parts = df["config"].str.split("__", expand=True)
df["F1_alg"] = parts[0].map({"nsgaiii": "NSGA-III", "rnsgaiii": "R-NSGA-III"})
df["F2_elit"] = parts[1].map({"elit_on": "ELIT_ON", "elit_off": "ELIT_OFF"})
df["F3_rest"] = parts[2].map({"penalty": "PENALTY", "kill": "KILL"})

fig, axes = plt.subplots(2, 4, figsize=(20, 9.8), dpi=300)
fig.suptitle(
    "Figura 3 — Perfis de Interação Fatorial 2×2 (Média ± 1 EP e Mediana, n = 10 por célula)\n"
    "Linha Superior: Algoritmo (F1) × Restrição (F3) | Linha Inferior: Algoritmo (F1) × Elitismo (F2)",
    fontsize=13.5, fontweight='bold', y=0.98
)

ind_meta = [
    ("hv", "Hypervolume (HV) [↑ Maior]",
     "F3 (Restrição): SS = 40.08%, p < 0.0001***\nF1×F3: SS = 1.05%, p = 0.4406",
     "F1 (Algoritmo): SS = 0.01%, p = 0.9387\nF1×F2: SS = 1.12%, p = 0.4256"),
    ("igd_plus", "IGD+ contra Z* [↓ Menor]",
     "F3 (Restrição): SS = 15.46%, p = 0.0124*\nF1×F3: SS = 3.93%, p = 0.1914",
     "F2 (Elitismo): SS = 1.93%, p = 0.3566\nF1×F2: SS = 3.34%, p = 0.2273"),
    ("ed_media", "ED Média ao Ideal [↓ Menor]",
     "F1 (Algoritmo): SS = 13.72%, p = 0.0219*\nF1×F3: SS = 0.94%, p = 0.5328",
     "F1 (Algoritmo): SS = 13.72%, p = 0.0219*\nF2: SS = 4.88% | F1×F2: SS = 2.28%"),
    ("spacing", "Spacing [↓ Menor = Mais Compacto]",
     "F3 (Restrição): SS = 49.89%, p < 0.0001***\nF1×F3: SS = 0.00%, p = 0.9795",
     "F2 (Elitismo): SS = 5.68%, p = 0.0454*\nF1×F2: SS = 0.07%, p = 0.8131"),
]

alg_levels = ["NSGA-III", "R-NSGA-III"]
x_pos = np.array([0, 1])

# Top row: F1 (Alg) x F3 (Rest: PENALTY vs KILL)
for col_idx, (ind, ylabel, anova_top, anova_bot) in enumerate(ind_meta):
    ax = axes[0, col_idx]
    for rest_lvl, color, marker, ls, offset in [
        ("PENALTY", "#0072B2", "o", "-", -0.03),
        ("KILL", "#D55E00", "s", "--", +0.03)
    ]:
        means, ses, meds = [], [], []
        for alg in alg_levels:
            sub = df[(df["F1_alg"] == alg) & (df["F3_rest"] == rest_lvl)][ind].to_numpy()
            means.append(sub.mean())
            ses.append(sub.std(ddof=1) / np.sqrt(len(sub)))
            meds.append(np.median(sub))
        ax.errorbar(
            x_pos + offset, means, yerr=ses, color=color, marker=marker, markersize=8,
            linewidth=2.3, linestyle=ls, capsize=5, capthick=1.5,
            label=f"{rest_lvl} (Média ± EP)"
        )
        # Plot median as hollow diamond connected by thin dotted line
        ax.plot(x_pos + offset, meds, color=color, marker='d', markersize=5.5,
                markerfacecolor='white', linestyle=':', linewidth=1.1, alpha=0.85,
                label=f"{rest_lvl} (Mediana)")
        for x_i, m_v in zip(x_pos + offset, means):
            ax.annotate(f"{m_v:.4f}", xy=(x_i, m_v), xytext=(0, 8 if rest_lvl=="PENALTY" else -13),
                        textcoords="offset points", ha='center', fontsize=8.0, fontweight='bold', color=color)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(alg_levels, fontsize=9.5, fontweight='bold')
    ax.set_xlim(-0.35, 1.35)
    ax.set_ylabel(ylabel, fontsize=9.2, fontweight='bold')
    ax.set_title(f"F1 × F3: {ylabel}\n{anova_top}", fontsize=9.2, fontweight='bold', pad=8)
    ax.grid(True, linestyle=':', alpha=0.35, color='#666666')
    sns.despine(ax=ax)

    # Add conditional effect callouts
    if ind == "hv":
        ax.set_ylim(0.73, 1.11)
        ax.text(0.5, 0.745, "Em R-NSGA-III: PEN vs KILL\nÂ12 = 0.86 (Grande), p = 0.0073**",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#EBF5FB', edgecolor='#0072B2', alpha=0.9))
    elif ind == "igd_plus":
        ax.set_ylim(0.08, 0.21)
        ax.text(0.5, 0.084, "Sob PENALTY: R-NSGA melhora -22.9%\n(Â12 = 0.69 Média); Sob KILL inverte!",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#FEF9E7', edgecolor='#D55E00', alpha=0.9))
    elif ind == "ed_media":
        ax.set_ylim(0.55, 0.73)
        ax.text(0.5, 0.556, "Sob PENALTY: R-NSGA vs NSGA\nÂ12 = 0.74 (Grande), Δ = -11.5%",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#EBF5FB', edgecolor='#0072B2', alpha=0.9))
    elif ind == "spacing":
        ax.set_ylim(0.025, 0.095)
        ax.text(0.5, 0.052, "Separação Total PEN vs KILL\nÂ12 = 0.01 (Grande), p < 0.0001***",
                ha='center', va='center', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#FDEDEC', edgecolor='#D55E00', alpha=0.9))

    if col_idx == 0:
        ax.legend(loc='upper left', fontsize=7.6, frameon=True, facecolor='white')

# Bottom row: F1 (Alg) x F2 (Elit: ELIT_ON vs ELIT_OFF)
for col_idx, (ind, ylabel, anova_top, anova_bot) in enumerate(ind_meta):
    ax = axes[1, col_idx]
    for elit_lvl, color, marker, ls, offset in [
        ("ELIT_ON", "#CC79A7", "^", "--", -0.03),
        ("ELIT_OFF", "#009E73", "o", "-", +0.03)
    ]:
        means, ses, meds = [], [], []
        for alg in alg_levels:
            sub = df[(df["F1_alg"] == alg) & (df["F2_elit"] == elit_lvl)][ind].to_numpy()
            means.append(sub.mean())
            ses.append(sub.std(ddof=1) / np.sqrt(len(sub)))
            meds.append(np.median(sub))
        ax.errorbar(
            x_pos + offset, means, yerr=ses, color=color, marker=marker, markersize=8,
            linewidth=2.3, linestyle=ls, capsize=5, capthick=1.5,
            label=f"{elit_lvl} (Média ± EP)"
        )
        ax.plot(x_pos + offset, meds, color=color, marker='d', markersize=5.5,
                markerfacecolor='white', linestyle=':', linewidth=1.1, alpha=0.85,
                label=f"{elit_lvl} (Mediana)")
        for x_i, m_v in zip(x_pos + offset, means):
            ax.annotate(f"{m_v:.4f}", xy=(x_i, m_v), xytext=(0, 8 if elit_lvl=="ELIT_OFF" else -13),
                        textcoords="offset points", ha='center', fontsize=8.0, fontweight='bold', color=color)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(alg_levels, fontsize=9.5, fontweight='bold')
    ax.set_xlim(-0.35, 1.35)
    ax.set_ylabel(ylabel, fontsize=9.2, fontweight='bold')
    ax.set_title(f"F1 × F2: {ylabel}\n{anova_bot}", fontsize=9.2, fontweight='bold', pad=8)
    ax.grid(True, linestyle=':', alpha=0.35, color='#666666')
    sns.despine(ax=ax)

    if ind == "hv":
        ax.set_ylim(0.80, 1.01)
        ax.text(0.5, 0.808, "Cruzamento Sinérgico:\nELIT_OFF eleva R-NSGA (+5.3%)",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.9))
    elif ind == "igd_plus":
        ax.set_ylim(0.10, 0.19)
        ax.text(0.5, 0.104, "Em R-NSGA-III: OFF vs ON\nReduz IGD+ em -21.1% (Â12 = 0.65)",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.9))
    elif ind == "ed_media":
        ax.set_ylim(0.53, 0.71)
        ax.text(0.5, 0.536, "Sob ELIT_OFF: R-NSGA vs NSGA\nÂ12 = 0.79 (Grande), p = 0.0312*",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.9))
    elif ind == "spacing":
        ax.set_ylim(0.035, 0.080)
        ax.text(0.5, 0.037, "ELIT_OFF melhora uniformidade\nem ambos algoritmos (p = 0.0454*)",
                ha='center', va='bottom', fontsize=7.8, bbox=dict(boxstyle='round,pad=0.25', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.9))

    if col_idx == 0:
        ax.legend(loc='upper left', fontsize=7.6, frameon=True, facecolor='white')

plt.subplots_adjust(left=0.05, right=0.98, top=0.89, bottom=0.06, hspace=0.38, wspace=0.24)
fig.savefig("fig3_interacoes_fatoriais_2x2.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig3_interacoes_fatoriais_2x2.png:", os.path.getsize("fig3_interacoes_fatoriais_2x2.png"), "bytes")

In [ ]:
with open("jpetstore_numMS4.json", "r", encoding="utf-8") as f:
    gen0_data = json.load(f)

zstar = np.array(meta["fronteira_referencia"], dtype=float)
z_ed = np.linalg.norm(zstar, axis=1)

lim_min = np.array(meta["limites_normalizacao"]["min"], dtype=float)
lim_max = np.array(meta["limites_normalizacao"]["max"], dtype=float)

g0_raw = np.array([m["value"] for m in gen0_data[0]["candidates"][0]["metrics"]], dtype=float)
g0_norm = (g0_raw - lim_min) / (lim_max - lim_min)

# Exact verified best-tradeoff points in Z* by matching ed_min of runs
# C4 Rep 3: Z*[55] (ED = 0.299792), C4 Rep 2: Z*[79] (ED = 0.303662), C4 Rep 5: Z*[97] (ED = 0.333945), C4 Rep 1: Z*[152] (ED = 0.374845)
# C3 Rep 3: Z*[32] (ED = 0.318985), C2 Rep 3: Z*[166] (ED = 0.326073), C1 Rep 4: Z*[151] (ED = 0.369608), C5 Rep 2 (KILL): Z*[164] (ED = 0.470251)
c4_excl_mask = z_ed < 0.318985  # 12 points 100% exclusive to C4
pen_excl_mask = (z_ed >= 0.318985) & (z_ed < 0.396786)  # 76 additional points 100% exclusive to PENALTY (total 88 < 0.396786)
outer_mask = z_ed >= 0.396786  # 137 points in outer region (where KILL can also reach)

x_axes = np.arange(5)
obj_labels = [
    "Coupling\n[90.0 – 342.0]",
    "Cohesion⁻¹ (1/v)\n[0.4076 – 18.2058]",
    "Overhead\n[192.0 – 2352.0]",
    "Functionality⁻¹ (1/v)\n[0.2660 – 0.3929]",
    "ReusePerMS⁻¹ (1/v)\n[1.0000 – 1.3333]"
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 11.8), dpi=300)
fig.suptitle(
    "Figura 4 — Coordenadas Paralelas e Perfis de Trade-off Arquitetural no Espaço Normalizado Pooled [0, 1]⁵\n"
    "(Convenção de Minimização Estrita: 0.00 = Ideal Teórico em todos os 5 Objetivos | P1-Reflective-Architect × JPetStore)",
    fontsize=13.2, fontweight='bold', y=0.98
)

# ------------------------------------------------------------------------------
# PAINEL SUPERIOR: Trajetórias Individuais de PF_known / Z* e Âncoras por Config
# ------------------------------------------------------------------------------
# 1. Background outer solutions (ED >= 0.3968)
for row in zstar[outer_mask]:
    ax1.plot(x_axes, row, color='#999999', alpha=0.16, linewidth=0.9, zorder=1)

# 2. PENALTY-exclusive solutions (0.3190 <= ED < 0.3968)
for row in zstar[pen_excl_mask]:
    ax1.plot(x_axes, row, color='#56B4E9', alpha=0.25, linewidth=1.1, zorder=2)

# 3. C4-exclusive core solutions (ED < 0.3190, n=12)
for row in zstar[c4_excl_mask]:
    ax1.plot(x_axes, row, color='#009E73', alpha=0.45, linewidth=1.5, zorder=3)

# 4. Highlighted verified best-tradeoff representatives per configuration
ax1.plot(x_axes, zstar[164], color='#D55E00', linewidth=2.5, linestyle='-', marker='s', markersize=6.5, zorder=6,
         label="Representante KILL em Z* — C5 Rep 2 (NSGA|ON|KILL, ED_min = 0.4703)")
ax1.plot(x_axes, zstar[151], color='#0072B2', linewidth=2.4, linestyle='-', marker='o', markersize=6.5, zorder=7,
         label="Melhor Trade-off C1 — C1 Rep 4 (NSGA|ON|PEN, ED_min = 0.3696)")
ax1.plot(x_axes, zstar[166], color='#2E86C1', linewidth=2.1, linestyle='-.', marker='^', markersize=6.0, zorder=7,
         label="Melhor Trade-off C2 — C2 Rep 3 (NSGA|OFF|PEN, ED_min = 0.3261)")
ax1.plot(x_axes, zstar[32], color='#8E44AD', linewidth=2.1, linestyle=':', marker='v', markersize=6.0, zorder=7,
         label="Melhor Trade-off C3 — C3 Rep 3 (R-NSGA|ON|PEN, ED_min = 0.3190)")
ax1.plot(x_axes, zstar[55], color='#009E73', linewidth=3.2, linestyle='-', marker='D', markersize=7.5, markeredgecolor='black', zorder=9,
         label="Ótimo Global C4* — C4 Rep 3 (R-NSGA|OFF|PEN, ED_min = 0.2998 — #1 de Z*)")
ax1.plot(x_axes, g0_norm, color='#111111', linewidth=2.8, linestyle='--', marker='X', markersize=8.0, zorder=10,
         label="Baseline Geração 0 (t = 0, 132 indivíduos determinísticos idênticos, ED = 1.4519)")

# Dummy handles for background bands
ax1.plot([], [], color='#009E73', alpha=0.6, linewidth=2.0, label="Núcleo Exclusivo C4 em Z* (ED < 0.3190, n = 12 soluções)")
ax1.plot([], [], color='#56B4E9', alpha=0.5, linewidth=1.8, label="Zona Exclusiva PENALTY em Z* (0.3190 ≤ ED < 0.3968, n = 76 soluções)")
ax1.plot([], [], color='#999999', alpha=0.5, linewidth=1.5, label="Região Externa de Z* (ED ≥ 0.3968, n = 137 soluções de extremo trade-off)")

for x_i in x_axes:
    ax1.axvline(x_i, color='#333333', linewidth=1.0, alpha=0.6, zorder=0)

ax1.set_xticks(x_axes)
ax1.set_xticklabels(obj_labels, fontsize=9.5, fontweight='bold')
ax1.set_xlim(-0.15, 4.15)
ax1.set_ylim(-0.04, 1.06)
ax1.set_ylabel("Valor Normalizado Pooled [0 = Ideal, 1 = Pior]", fontsize=9.8, fontweight='bold')
ax1.set_title(
    "(a) Topologia Multidimensional da Fronteira de Pareto (Z*, n = 225) e Soluções de Melhor Compromisso Verificadas por Configuração",
    fontsize=10.8, fontweight='bold', pad=8
)
ax1.grid(axis='y', linestyle=':', alpha=0.35, color='#666666')
ax1.legend(loc='upper right', fontsize=8.0, ncol=2, frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.94)
sns.despine(ax=ax1)

# ------------------------------------------------------------------------------
# PAINEL INFERIOR: Perfis de Mediana e Bandas Interquartis (Q25–Q75)
# ------------------------------------------------------------------------------
# Groups to compare in Panel B:
# 1. C4 best-tradeoff verified runs in Z* + C4 exclusive core (n=15 solutions in Z*)
c4_verified_indices = list(set(np.where(c4_excl_mask)[0].tolist() + [55, 79, 97, 152]))
c4_set = zstar[c4_verified_indices]

# 2. All PENALTY-exclusive solutions in Z* (ED < 0.396786, n=88)
pen_set = zstar[z_ed < 0.396786]

# 3. Outer / KILL-accessible zone (ED >= 0.396786, n=137)
kill_zone_set = zstar[z_ed >= 0.396786]

# 4. Full Z* reference frontier (n=225)
full_z_set = zstar

profiles = [
    ("Fronteira Global Z* Completa (n = 225)", full_z_set, "#555555", "-.", "o", 0.12),
    ("Zona Externa / Acessível a KILL (ED ≥ 0.3968, n = 137)", kill_zone_set, "#D55E00", "--", "s", 0.16),
    ("Zona Exclusiva PENALTY (ED < 0.3968, n = 88)", pen_set, "#0072B2", "-", "^", 0.18),
    ("Núcleo C4* (R-NSGA|OFF|PEN em Z*, n = 15)", c4_set, "#009E73", "-", "D", 0.25),
]

for label, mat, color, ls, marker, band_alpha in profiles:
    med = np.median(mat, axis=0)
    q25 = np.percentile(mat, 25, axis=0)
    q75 = np.percentile(mat, 75, axis=0)
    ax2.plot(x_axes, med, color=color, linestyle=ls, marker=marker, markersize=7.0, linewidth=2.5, label=f"{label} [Mediana + IQR]")
    ax2.fill_between(x_axes, q25, q75, color=color, alpha=band_alpha)

# Also plot Gen 0 and Ideal (0,0,0,0,0)
ax2.plot(x_axes, g0_norm, color='#111111', linestyle='--', marker='X', markersize=7.5, linewidth=2.3,
         label="Baseline Geração 0 (t = 0): [0.9563, 0.3162, 0.9074, 0.5195, 0.0000]")
ax2.plot(x_axes, np.zeros(5), color='#2ECC71', linestyle=':', linewidth=2.0,
         label="Vetor Ideal Teórico [0, 0, 0, 0, 0]")

# Annotate C4 median values
c4_med = np.median(c4_set, axis=0)
for x_i, val in enumerate(c4_med):
    ax2.annotate(f"C4: {val:.3f}", xy=(x_i, val), xytext=(0, -14 if x_i != 4 else 10),
                 textcoords="offset points", ha='center', fontsize=8.2, fontweight='bold', color='#004D38',
                 bbox=dict(boxstyle='round,pad=0.15', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.9))

for x_i in x_axes:
    ax2.axvline(x_i, color='#333333', linewidth=1.0, alpha=0.6, zorder=0)

ax2.text(0.5, 0.78, "Conflito Estrutural Primário:\nReduzir Coupling (-83% vs Gen 0)\ne Overhead (-86% vs Gen 0) exige\nbalancear Cohesion⁻¹ e Functionality⁻¹",
         ha='center', va='center', fontsize=8.6, fontweight='bold', color='#222222',
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#FEF9E7', edgecolor='#B7950B', alpha=0.94))

ax2.set_xticks(x_axes)
ax2.set_xticklabels(obj_labels, fontsize=9.5, fontweight='bold')
ax2.set_xlim(-0.15, 4.15)
ax2.set_ylim(-0.06, 1.06)
ax2.set_ylabel("Mediana e Intervalo Interquartil (Q25–Q75)", fontsize=9.8, fontweight='bold')
ax2.set_title(
    "(b) Perfil Comparativo de Medianas e Bandas Interquartis (Q25–Q75) por Eixo Objetivo: Núcleo C4 vs. PENALTY vs. Zona KILL vs. Gen 0",
    fontsize=10.8, fontweight='bold', pad=8
)
ax2.grid(axis='y', linestyle=':', alpha=0.35, color='#666666')
ax2.legend(loc='upper right', fontsize=8.2, ncol=2, frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.94)
sns.despine(ax=ax2)

plt.subplots_adjust(left=0.06, right=0.98, top=0.90, bottom=0.06, hspace=0.32)
fig.savefig("fig4_coordenadas_paralelas_PF.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig4_coordenadas_paralelas_PF.png:", os.path.getsize("fig4_coordenadas_paralelas_PF.png"), "bytes")

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 7.6), dpi=300)
fig.suptitle(
    "Figura 5 — Diagnóstico do Paradoxo Geométrico do Spacing e Deslocamento de Convergência (Geração 0 → Geração 999)",
    fontsize=13.5, fontweight='bold', y=0.98
)

# ------------------------------------------------------------------------------
# PAINEL ESQUERDO: Scatter Plot Spacing (X) vs. HV (Y) (N = 40)
# ------------------------------------------------------------------------------
# Regression line across all N=40
sns.regplot(
    data=df, x="spacing", y="hv", ax=ax_left, scatter=False,
    color='#444444', line_kws={"linestyle": "--", "linewidth": 1.8, "label": "Tendência Global (N=40): r = +0.6708, ρ = +0.6722***"}
)

# Plot points stratified by F3_rest x F1_alg, highlighting C4
for _, row in df.iterrows():
    is_c4 = (row["config"] == "rnsgaiii__elit_off__penalty")
    is_pen = (row["F3_rest"] == "PENALTY")
    is_rnsga = (row["F1_alg"] == "R-NSGA-III")

    color = "#009E73" if is_c4 else ("#0072B2" if is_pen else "#D55E00")
    marker = "^" if is_rnsga else "o"
    size = 115 if is_c4 else 72
    edge_w = 1.8 if is_c4 else 1.0

    ax_left.scatter(
        row["spacing"], row["hv"], color=color, marker=marker, s=size,
        edgecolor='black', linewidth=edge_w, alpha=0.92, zorder=6 if is_c4 else 4
    )
    if is_c4:
        ax_left.annotate(
            f"C4-R{row['repetition']}", (row["spacing"], row["hv"]),
            xytext=(5, 5), textcoords="offset points", fontsize=7.8, fontweight='bold', color='#004D38'
        )

# Add shaded regions for KILL vs PENALTY
ax_left.axvspan(0.020, 0.050, ymin=0.0, ymax=0.52, color='#D55E00', alpha=0.08, zorder=1)
ax_left.text(0.035, 0.635, "Zona de Colapso Espacial (KILL)\nBaixo Spacing (0.0350) + Baixo HV (0.8098)\nFronteira curta e retraída longe de Z*",
             ha='center', va='bottom', fontsize=8.2, fontweight='bold', color='#A04000',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#FDEDEC', edgecolor='#D55E00', alpha=0.92))

ax_left.text(0.105, 1.15, "Zona de Expansão de Pareto (PENALTY)\nAlto HV (0.9928) + Maior Extensão Espacial\nC4 otimiza HV (1.0643) com Spacing moderado (0.0647)",
             ha='center', va='center', fontsize=8.2, fontweight='bold', color='#004D38',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#D1F2EB', edgecolor='#009E73', alpha=0.92))

# Custom legend for Left Panel
leg_handles_left = [
    plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='#009E73', markeredgecolor='black', markersize=10, markeredgewidth=1.6, label='C4* (R-NSGA-III | ELIT_OFF | PENALTY, n=5)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#0072B2', markeredgecolor='black', markersize=8, label='NSGA-III | PENALTY (C1, C2, n=10)'),
    plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='#0072B2', markeredgecolor='black', markersize=8, label='R-NSGA-III | ELIT_ON | PENALTY (C3, n=5)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#D55E00', markeredgecolor='black', markersize=8, label='NSGA-III | KILL (C5, C6, n=10)'),
    plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='#D55E00', markeredgecolor='black', markersize=8, label='R-NSGA-III | KILL (C7, C8, n=10)'),
    plt.Line2D([0], [0], color='#444444', linestyle='--', linewidth=1.8, label='Regressão Linear (Spearman ρ = +0.6722, p = 2.03e-6)'),
]
ax_left.set_xlabel("Spacing (Desvio-Padrão das Distâncias de Manhattan ao Vizinho Mais Próximo)", fontsize=9.8, fontweight='bold')
ax_left.set_ylabel("Hypervolume (HV — Cobertura e Convergência no Espaço [0, 1]⁵)", fontsize=9.8, fontweight='bold')
ax_left.set_title(
    "(a) Resolução do Paradoxo do Spacing (N = 40 Execuções)\nCorrelação Positiva Forte (ρ = +0.6722, p = 2.03×10⁻⁶) Comprova Estreitamento Sob KILL",
    fontsize=10.5, fontweight='bold', pad=8
)
ax_left.set_xlim(0.018, 0.160)
ax_left.set_ylim(0.60, 1.28)
ax_left.grid(True, linestyle=':', alpha=0.35, color='#666666')
ax_left.legend(handles=leg_handles_left, loc='lower right', fontsize=8.0, frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.95)
sns.despine(ax=ax_left)

# ------------------------------------------------------------------------------
# PAINEL DIREITO: Deslocamento de Convergência Normalizado (Gen 0 [t=0] -> Gen 999)
# ------------------------------------------------------------------------------
# Metrics to compare from Gen 0 (t=0) to Gen 999 (t=final):
# 5 Normalized Objectives (Coupling, Cohesion⁻¹, Overhead, Functionality⁻¹, Reuse⁻¹)
# + 3 Global Convergence Metrics (ED_média, ED_min, IGD+)
# All 8 metrics are in MINIMIZATION direction (lower = better)!
metric_labels_conv = [
    "Coupling\n(Obj 1)",
    "Cohesion⁻¹\n(Obj 2)",
    "Overhead\n(Obj 3)",
    "Func.⁻¹\n(Obj 4)",
    "Reuse⁻¹\n(Obj 5)",
    "ED_média\nao Ideal",
    "ED_min\nao Ideal",
    "IGD+\naté Z*"
]

# Gen 0 values (exact from Phase 1):
# g0_norm = [0.956349, 0.316206, 0.907407, 0.519533, 0.000000]
# ED_média(Gen0) = ED_min(Gen0) = 1.451858, IGD+(Gen0) = 1.105329
gen0_profile = np.array([g0_norm[0], g0_norm[1], g0_norm[2], g0_norm[3], g0_norm[4], 1.451858, 1.451858, 1.105329])

# Gen 999 profiles (combining objective medians in Z* strata + exact indicator medians across runs):
kill_med_obj = np.median(kill_zone_set, axis=0)
kill_med_ind = [
    df[df["F3_rest"]=="KILL"]["ed_media"].median(),
    df[df["F3_rest"]=="KILL"]["ed_min"].median(),
    df[df["F3_rest"]=="KILL"]["igd_plus"].median()
]
kill_profile = np.concatenate([kill_med_obj, kill_med_ind])

pen_med_obj = np.median(pen_set, axis=0)
pen_med_ind = [
    df[df["F3_rest"]=="PENALTY"]["ed_media"].median(),
    df[df["F3_rest"]=="PENALTY"]["ed_min"].median(),
    df[df["F3_rest"]=="PENALTY"]["igd_plus"].median()
]
pen_profile = np.concatenate([pen_med_obj, pen_med_ind])

c4_med_obj = np.median(c4_set, axis=0)
c4_med_ind = [
    df[df["config"]=="rnsgaiii__elit_off__penalty"]["ed_media"].median(),
    df[df["config"]=="rnsgaiii__elit_off__penalty"]["ed_min"].median(),
    df[df["config"]=="rnsgaiii__elit_off__penalty"]["igd_plus"].median()
]
c4_profile = np.concatenate([c4_med_obj, c4_med_ind])

x_conv = np.arange(len(metric_labels_conv))

ax_right.plot(x_conv, gen0_profile, color='#111111', linestyle='--', marker='X', markersize=8.0, linewidth=2.5,
              label="Baseline Geração 0 (t = 0, População Inicial Determinística, HV = 0.0138)")
ax_right.plot(x_conv, kill_profile, color='#D55E00', linestyle='-.', marker='s', markersize=6.8, linewidth=2.2,
              label="Gen 999 — Bloco KILL (C5–C8, HV Med = 0.8336, Ganho HV = 60.2×)")
ax_right.plot(x_conv, pen_profile, color='#0072B2', linestyle='-', marker='o', markersize=6.8, linewidth=2.2,
              label="Gen 999 — Bloco PENALTY (C1–C4, HV Med = 0.9940, Ganho HV = 71.8×)")
ax_right.plot(x_conv, c4_profile, color='#009E73', linestyle='-', marker='D', markersize=7.8, markeredgecolor='black', linewidth=2.8,
              label="Gen 999 — Campeã C4* (R-NSGA|OFF|PEN, HV Med = 1.0551, Ganho HV = 76.2×)")

# Fill area of convergence improvement between Gen 0 and C4
ax_right.fill_between(x_conv, c4_profile, gen0_profile, color='#009E73', alpha=0.10, label="Magnitude de Convergência Alcançada por C4* (t=0 → t=999)")

# Annotate percentage reductions from Gen 0 to C4*
for idx in [0, 1, 2, 3, 5, 6, 7]:
    g0_v = gen0_profile[idx]
    c4_v = c4_profile[idx]
    pct_red = (c4_v - g0_v) / g0_v * 100.0
    # Draw vertical arrow of improvement
    ax_right.annotate(
        "", xy=(idx, c4_v + 0.02), xytext=(idx, g0_v - 0.02),
        arrowprops=dict(arrowstyle="->", color='#009E73', lw=1.2, ls=':')
    )
    y_text = (g0_v + c4_v) / 2.0
    ax_right.text(
        idx + 0.06, y_text, f"{pct_red:.1f}%",
        ha='left', va='center', fontsize=8.0, fontweight='bold', color='#004D38',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='#009E73', alpha=0.88)
    )

ax_right.axvline(4.5, color='#444444', linestyle='--', linewidth=1.3, alpha=0.8)
ax_right.text(2.0, 1.48, "5 Funções Objetivo Normalizadas [0, 1]", ha='center', va='top', fontsize=8.6, fontweight='bold', color='#333333')
ax_right.text(6.0, 1.48, "Indicadores Globais de Convergência", ha='center', va='top', fontsize=8.6, fontweight='bold', color='#333333')

ax_right.set_xticks(x_conv)
ax_right.set_xticklabels(metric_labels_conv, fontsize=8.8, fontweight='bold')
ax_right.set_xlim(-0.35, 7.65)
ax_right.set_ylim(-0.05, 1.56)
ax_right.set_ylabel("Valor Normalizado / Distância (↓ Menor é Melhor)", fontsize=9.8, fontweight='bold')
ax_right.set_title(
    "(b) Salto de Convergência Evolutiva: Baseline Geração 0 (t = 0) vs. Fronteiras Finais (Gen 999)\n"
    "Reduções de -61.8% em ED_média, -77.0% em ED_min e -92.0% em IGD+ pela Configuração C4*",
    fontsize=10.5, fontweight='bold', pad=8
)
ax_right.grid(True, linestyle=':', alpha=0.35, color='#666666')
ax_right.legend(loc='upper left', fontsize=7.8, frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.95)
sns.despine(ax=ax_right)

plt.subplots_adjust(left=0.05, right=0.98, top=0.88, bottom=0.10, wspace=0.20)
fig.savefig("fig5_paradoxo_spacing_e_convergencia_gen0.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved fig5_paradoxo_spacing_e_convergencia_gen0.png:", os.path.getsize("fig5_paradoxo_spacing_e_convergencia_gen0.png"), "bytes")

In [ ]:
expected_figs = [
    "fig1_distribuicao_indicadores.png",
    "fig2_heatmaps_efeito_A12_fdr.png",
    "fig3_interacoes_fatoriais_2x2.png",
    "fig4_coordenadas_paralelas_PF.png",
    "fig5_paradoxo_spacing_e_convergencia_gen0.png",
]
for fname in expected_figs:
    exists = os.path.exists(fname)
    size_kb = os.path.getsize(fname) / 1024.0 if exists else 0.0
    assert exists and size_kb > 50.0, f"Falha na verificação de {fname}"
    print(f"[OK] {fname:45s} | {size_kb:8.1f} KB (> 50 KB)")